# Thermal Image Restoration with Deep Image Prior (DIP)
### Module 01: Environment Setup and Single-Image Restoration Pipeline
**Architecture:** 5-Level U-Net Generator with 1x1 Skip Projections (DIP Paper Standard)  
**Hardware:** Google Colab GPU (NVIDIA T4 / V100 / A100)  
**Dataset requirement:** None (Unsupervised single-image optimization via network inductive bias)

In [ ]:
# 1. Hardware Verification
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")
else:
    print("WARNING: GPU not detected. Running on CPU will be slower.")

In [ ]:
# 2. Clone Repository and Install DIP Core Package
import os, sys
from pathlib import Path

# If running directly inside Google Colab root
if not Path("ml/dip_core").exists():
    !git clone https://github.com/likhith-naik01/DIP_MP.git
    %cd DIP_MP

# Install the ml package in editable mode
!pip install -q -e ml/
print("dip_core package installed and ready for import.")

In [ ]:
# 3. Mount Google Drive (Optional for saving results)
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/DIP_outputs/single_restoration')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Output folder set to: {DRIVE_OUTPUT_DIR}")
except Exception as e:
    DRIVE_OUTPUT_DIR = Path('./outputs/single_restoration')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive not mounted. Using local storage: {DRIVE_OUTPUT_DIR}")

In [ ]:
# 4. Select or Upload Thermal Image
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

sample_dir = Path("ml/sample_images")
available_samples = list(sample_dir.glob("*.png"))
print("Available sample thermal images in repo:")
for idx, p in enumerate(available_samples):
    print(f"  [{idx}] {p.name}")

# Select first sample by default
selected_image_path = available_samples[0] if available_samples else None
print(f"\nUsing image: {selected_image_path}")

img = Image.open(selected_image_path).convert("L")
plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="inferno")
plt.title(f"Target Thermal Image: {selected_image_path.name}")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)
plt.show()

In [ ]:
# 5. Execute Deep Image Prior Restoration
from dip_core.config import DipTrainingConfig, RestorationTask, DamageConfig
from dip_core.pipeline.restoration_pipeline import run_restoration
from dip_core.pipeline.damage_simulation import simulate_thermal_damage

# Step A: Simulate thermal sensor degradation (Dead pixels + Dead sensor streak)
img_np = np.array(img, dtype=np.float32) / 255.0
damage_cfg = DamageConfig(
    dropout_ratio=0.015,
    num_line_holes=2,
    seed=42
)
degraded_np, mask_np = simulate_thermal_damage(img_np, damage_cfg)

# Step B: Configure DIP Training
train_cfg = DipTrainingConfig(
    iterations=1500,
    learning_rate=0.01,
    reg_noise_std=0.03,
    save_every=100,
    log_every=100,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Starting DIP Restoration on device: {train_cfg.device} ...")
output = run_restoration(
    image_input=degraded_np,
    mask_input=mask_np,
    task=RestorationTask.INPAINT,
    model_name="unet",
    training_config=train_cfg,
    ground_truth=img_np
)

print(f"\nOptimization finished in {output.runtime_seconds:.1f} seconds.")
print(f"PSNR Improvement: {output.metrics.get('initial_psnr', 0.0):.2f} dB -> {output.metrics.get('psnr', 0.0):.2f} dB")
print(f"SSIM Improvement: {output.metrics.get('initial_ssim', 0.0):.4f} -> {output.metrics.get('ssim', 0.0):.4f}")

In [ ]:
# 6. Side-by-Side Visual Comparison and Loss Curve
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img_np, cmap="inferno")
axes[0].set_title("Ground Truth Thermal", fontsize=12)
axes[0].axis("off")

axes[1].imshow(degraded_np, cmap="inferno")
axes[1].set_title("Degraded (Dead Pixels + Streaks)", fontsize=12)
axes[1].axis("off")

axes[2].imshow(mask_np, cmap="gray")
axes[2].set_title("Valid Pixel Mask", fontsize=12)
axes[2].axis("off")

axes[3].imshow(output.restored_array, cmap="inferno")
axes[3].set_title(f"U-Net DIP Restored\nPSNR: {output.metrics.get('psnr', 0.0):.2f} dB", fontsize=12)
axes[3].axis("off")

plt.tight_layout()
save_vis_path = DRIVE_OUTPUT_DIR / "single_restoration_comparison.png"
plt.savefig(save_vis_path, dpi=200, bbox_inches="tight")
plt.show()

# Convergence Loss Plot
plt.figure(figsize=(9, 4))
plt.plot(output.losses, color="#00e5ff", lw=2, label="Masked MSE Loss")
plt.yscale("log")
plt.grid(True, linestyle="--", alpha=0.5)
plt.title("U-Net Deep Image Prior Optimization Convergence", fontsize=13)
plt.xlabel("Iteration Step", fontsize=11)
plt.ylabel("Loss (Log Scale)", fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT_DIR / "loss_curve.png", dpi=200)
plt.show()

In [ ]:
# 7. Export Outputs to Google Drive
output_restored_img = Image.fromarray((output.restored_array * 255.0).clip(0, 255).astype(np.uint8))
output_restored_img.save(DRIVE_OUTPUT_DIR / "restored_thermal.png")

import json
metrics_path = DRIVE_OUTPUT_DIR / "restoration_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(output.metrics, f, indent=2)

print(f"Successfully exported all results to: {DRIVE_OUTPUT_DIR}")